# 01.6 — ADME Biogen Public: Results & Comparison (§5)

Split out of `01.5_adme_biogen_public_recreation.ipynb` to keep each notebook loadable.
This notebook contains **§5 — Results Comparison** only. It consumes the Section-4
checkpoint written by `01.5` (`section4_splits.pkl`, `section4_test_predictions.pkl`,
`section4_paper_recreation_results.csv`) — no modelling is recomputed here.

**Run order:** §0 setup → the loader cell below → §5.


## 0 — Setup

> **NOTE**: The paper's train/test split files are needed to exactly reproduce their results.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

RANDOM_STATE = 42

DATA_RAW   = Path('../data/raw')
DATA_PROC  = Path('../data/processed')
FIGURES    = Path('../figures')

ADME_FILE  = DATA_RAW / 'ADME_public_set_3521.csv'
SPLIT_FILE_1 = DATA_RAW / 'TODO_split_file.csv'  # TODO: update once split files are provided

SDF_DIR = DATA_RAW.parent / 'sdfs'
LOG_FILE = DATA_PROC / 'sdf_standardization.log'

SDF_FILES = {
    'HLM':   SDF_DIR / 'ADME_HLM.sdf',
    'MDR1':  SDF_DIR / 'ADME_MDR1_ER.sdf',
    'RLM':   SDF_DIR / 'ADME_RLM.sdf',
    'SOL':   SDF_DIR / 'ADME_Sol.sdf',
    'PPB_H': SDF_DIR / 'ADME_hPPB.sdf',
    'PPB_R': SDF_DIR / 'ADME_rPPB.sdf',
}

print('Setup complete')
print(f'ADME file exists: {ADME_FILE.exists()}')
print(f'Split files exist: {SPLIT_FILE_1.exists()}  <-- update path once available')

In [ ]:
import contextlib

from scipy.stats import gaussian_kde
from scipy import stats
import plotly.graph_objects as go

from rdkit import Chem, DataStructs
from rdkit.Chem import SDMolSupplier, rdMolDescriptors

from src.eda import missing_value_report
from src.features import morgan_fingerprints, rdmoldes, rdkit_2d_features, fcfp4_bit_vectors
from src.preprocessing import standardize

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.utils import shuffle
from sklearn.base import clone

from statsmodels.stats.anova import AnovaRM
from statsmodels.stats.multicomp import pairwise_tukeyhsd

from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
import seaborn as sns

import joblib

from src.hyperparams import PARAM_GRID_STAGES, param_base_MPNN, param_base_FCNN, FCNN_ARCHITECTURES
from src.models import (get_paper_models, tune_paper_model, tune_fcnn_architecture, model_validation,
                        load_eval_checkpoint, run_checkpointed_eval, invalidate_checkpoint,
                        FCNN, ChempropRegressor, tune_mpnn_hyperopt)
from src.metrics import mae

### 0.1 — Endpoint constants (relocated from §2/§3)

`ENDPOINTS`/`ENDPOINT_COLS`/`MODEL_ENDPOINTS` are the only names §5 needs that were
originally defined in the modelling sections. Copied here verbatim so this notebook is self-contained.


In [ ]:
ENDPOINTS = {
    'HLM':   'LOG HLM_CLint (mL/min/kg)',
    'MDR1':  'LOG MDR1-MDCK ER (B-A/A-B)',
    'SOL':   'LOG SOLUBILITY PH 6.8 (ug/mL)',
    'RLM':   'LOG RLM_CLint (mL/min/kg)',
    'PPB_H': 'LOG PLASMA PROTEIN BINDING (HUMAN) (% unbound)',
    'PPB_R': 'LOG PLASMA PROTEIN BINDING (RAT) (% unbound)',
}
ENDPOINT_COLS = list(ENDPOINTS.values())
MODEL_ENDPOINTS = ['HLM', 'MDR1', 'SOL', 'RLM']  # ppb_h/ppb_r not modelled


## 5 — Results Comparison

Compare our reproduced numbers against the paper's reported numbers.

### 5.pre — Load the Section-4 checkpoint (written by `01.5`)

`splits` is loaded straight from `section4_splits.pkl` — no need to recompute §3 featurization.
If these files are missing, run `01.5` sections 0–4 first.


In [ ]:
splits = joblib.load(DATA_PROC / 'section4_splits.pkl')
predictions = joblib.load(DATA_PROC / 'section4_test_predictions.pkl')
results_df = pd.read_csv(DATA_PROC / 'section4_paper_recreation_results.csv')
df = joblib.load(DATA_PROC / 'section4_df.pkl')  # raw cleaned df -- §5.7 Table 2 top block
base_df = results_df[results_df['arm'] == 'base'].copy()


### 5.0 — Dummy-data harness (dev only)

Lets Section 5's tables/plots be developed **before** the real overnight run exists. Set `USE_DUMMY_SECTION5 = True` to replace `results_df` / `predictions` / `base_df` with synthetic data that matches the real checkpoint schema exactly — all 9 models, MPNN under its `graph`/`graph_rdkit` featuresets, a `mpnn_mode` column, and 15-length `cv_scores` so the paired-ANOVA path (5.5) exercises correctly. It reads the **real** `splits` (so test-set sizes / SMILES are real) but never touches the on-disk checkpoint files. Set back to `False` to use the real data.

In [ ]:
USE_DUMMY_SECTION5 = False  # True -> develop Section 5 on synthetic data (real files untouched)

def make_dummy_checkpoint(splits, n_folds=15, seed=0):
    """Synthetic (results_df, predictions) matching the Section 4 schema, for developing Section 5."""
    from scipy.stats import pearsonr
    rng = np.random.default_rng(seed)
    CLASSICAL = ['RF', 'SVM', 'XGBoost', 'LightGBM', 'Lasso', 'BayesianRidge', 'FCNN']
    CLASSICAL_FS = ['fcfp4', 'rdkit', 'hybrid', 'ecfp4', 'hybrid_ecfp4']
    MPNN = [('MPNN1', 'graph'), ('MPNN2', 'graph_rdkit'), ('MPNN3', 'graph_rdkit2dnorm')]
    EPS = ['HLM', 'MDR1', 'SOL', 'RLM']
    preds, rows = {}, []

    def emit(ep, fs, model, y_test, mode=np.nan, base_r=0.7):
        yhat = np.asarray(y_test, float) + rng.normal(0, 0.5, size=len(y_test))
        cv = np.clip(rng.normal(base_r, 0.05, size=n_folds), -1, 1)
        preds[(ep, fs, model, 'base')] = {'y_test': np.asarray(y_test, float),
                                          'y_pred_test': yhat, 'cv_scores': cv,
                                          **({'mpnn_mode': mode} if isinstance(mode, str) else {})}
        r = pearsonr(np.asarray(y_test, float), yhat)[0]
        rows.append({'endpoint': ep, 'featureset': fs, 'model': model, 'arm': 'base',
                     'mpnn_mode': mode, 'Pearson_r_CV': float(np.nanmean(cv)),
                     'Pearson_r_test': float(r), 'MAE': float(np.mean(np.abs(np.asarray(y_test, float) - yhat)))})

    for ep in EPS:
        for fs in CLASSICAL_FS:
            for m in CLASSICAL:
                emit(ep, fs, m, splits[(ep, fs)]['y_test'])
        for m, fs in MPNN:  # MPNN reads SMILES/y from the fcfp4 split (identical across featuresets)
            emit(ep, fs, m, splits[(ep, 'fcfp4')]['y_test'], mode='full', base_r=0.65)
    return pd.DataFrame(rows), preds

if USE_DUMMY_SECTION5:
    results_df, predictions = make_dummy_checkpoint(splits)
    base_df = results_df[results_df['arm'] == 'base'].copy()
    print(f'DUMMY Section 5 data in use: results_df {results_df.shape}, {len(predictions)} prediction keys')

### 5.1 — MAE summary table

Paper's domain-of-applicability metric. All values below are from the `'base'` arm (paper's
default hyperparameters) — the `'tuned'` arm hasn't been run yet (see Section 5.6).

In [ ]:
base_df = results_df[results_df['arm'] == 'base'].copy()

# dropna=False: without it, pivot_table silently DROPS any column that's 100% NaN across every
# endpoint (Lasso+fcfp4 is all-NaN for Pearson_r_test, see 5.2)
mae_table = base_df.pivot_table(index='endpoint', columns=['model', 'featureset'], values='MAE', dropna=False)
mae_table = mae_table.reindex(index=['HLM', 'MDR1', 'SOL', 'RLM'])
# MAE itself is not NaN for Lasso+fcfp4 (Section 4.2a)
mae_table.round(3)


### 5.2 — Pearson r summary table

The paper's headline reported metric. Using `Pearson_r_test` (held-out test set, from the model
fit on the full training set)

In [ ]:
pearson_table = base_df.pivot_table(index='endpoint', columns=['model', 'featureset'], values='Pearson_r_test', dropna=False)
pearson_table = pearson_table.reindex(index=['HLM', 'MDR1', 'SOL', 'RLM'])
print("NaN = Lasso+fcfp4 collapse (Section 4.2a) -- expected, not missing data.")
pearson_table.round(3)

### 5.3 -- Pearson r boxplots per model, per endpoint (Papers Figure 1)

Matches the paper's model-comparison figure style (one box per model, spread from repeated
held-out evaluation). Box = that model's 15 RepeatedKFold
CV-fold Pearson r scores (persisted in Section 4.2 as `cv_scores`), using a single representative
featureset (`BOXPLOT_FEATURESET`, defaults to `'hybrid'`) per model.

Not an exact analog of the paper's figure -- their spread comes from 20 independent in-house
datasets; ours comes from CV-fold resampling of a single train/test split -- but it's the closest
real distribution available, and produces the same box-and-whisker shape.


In [ ]:
BOXPLOT_FEATURESET = 'hybrid'  # single representative featureset for this model-comparison boxplot
BOXPLOT_MODEL_FS = {'MPNN1': 'graph', 'MPNN2': 'graph_rdkit', 'MPNN3': 'graph_rdkit2dnorm'}  # MPNN has no hybrid preds; shown at its graph featureset
model_names = ['RF', 'SVM', 'XGBoost', 'LightGBM', 'FCNN', 'MPNN1', 'MPNN2', 'MPNN3', 'Lasso', 'BayesianRidge']
endpoints_list = ['HLM', 'RLM', 'MDR1', 'SOL']

fig, axes = plt.subplots(2, 2, figsize=(11, 9), sharey=False)
axes = axes.flatten()

for ax, ep in zip(axes, endpoints_list):
    groups = []
    for m in model_names:
        pred = predictions.get((ep, BOXPLOT_MODEL_FS.get(m, BOXPLOT_FEATURESET), m, 'base'))
        scores = np.asarray(pred['cv_scores']) if pred is not None else np.array([])
        scores = scores[~np.isnan(scores)]
        groups.append(scores)
    ax.boxplot(groups, labels=model_names)
    ax.set_title(ep, fontsize=11)
    ax.set_ylabel('Pearson r (CV folds)')
    ax.set_ylim(0.4, 1.0)
    ax.grid(alpha=0.3, axis='y')
    ax.tick_params(axis='x', rotation=30)

fig.suptitle(f'Pearson r per model, per endpoint (15 CV-fold scores, featureset={BOXPLOT_FEATURESET}; MPNN at its graph featureset)', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES / 'section5_pearson_r_boxplot_per_endpoint.png', dpi=150, bbox_inches='tight')
plt.show()


### 5.3b -- Effect of molecular representation on model performance (Papers Figure 5)

Compares the fingerprint / descriptor / hybrid representations for RF, LightGBM and FCNN on HLM and
MDR1. Box = the same 15 CV-fold Pearson r scores per (model, featureset) used in Section 5.3.

Featuresets shown: `fcfp4`, `ecfp4`, `rdkit`, `hybrid` (fcfp4+rdMolDes), `hybrid_ecfp4` (ecfp4+rdMolDes).
ECFP4 and its hybrid were added as modelling representations for this comparison (see DECISIONS.md).

In [ ]:
REPR_MODELS = ['RF', 'LightGBM', 'FCNN']#, 'BayesianRidge']
REPR_FEATURESETS = ['fcfp4', 'ecfp4', 'rdkit', 'hybrid', 'hybrid_ecfp4']
REPR_ENDPOINTS = ['HLM', 'MDR1']
FS_COLORS = {'fcfp4': 'white', 'ecfp4': '#e0e0e0', 'rdkit': 'black', 'hybrid': 'lightgrey', 'hybrid_ecfp4': 'dimgrey'}

fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=False)

for ax, ep in zip(axes, REPR_ENDPOINTS):
    positions, groups, box_fs = [], [], []
    xticks, xticklabels = [], []
    pos = 0

    for m in REPR_MODELS:
        cluster_start = pos
        for fs in REPR_FEATURESETS:
            pred = predictions.get((ep, fs, m, 'base'))
            scores = np.asarray(pred['cv_scores']) if pred is not None else np.array([])
            scores = scores[~np.isnan(scores)]
            groups.append(scores)
            positions.append(pos)
            box_fs.append(fs)
            pos += 1
        xticks.append((cluster_start + pos - 1) / 2)
        xticklabels.append(m)
        pos += 1  # gap between model clusters

    bp = ax.boxplot(groups, positions=positions, widths=0.6, patch_artist=True)
    for patch, fs in zip(bp['boxes'], box_fs):
        patch.set_facecolor(FS_COLORS[fs])

    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels)
    ax.set_title(ep, fontsize=11)
    ax.set_ylabel('Pearson r (CV folds)')
    ax.set_ylim(0.4, 1.0)
    ax.grid(alpha=0.3, axis='y')

legend_handles = [Patch(facecolor=FS_COLORS[fs], edgecolor='black', label=fs) for fs in REPR_FEATURESETS]
axes[-1].legend(handles=legend_handles, loc='upper right', fontsize=8)

fig.suptitle('Effect of molecular representation on model performance (15 CV-fold Pearson r scores)', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES / 'section5_representation_effect_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()


### 5.4 -- Similarity-binned MAE (RF & LightGBM, HLM & MDR1), (Papers Figure 6)

Dice and Tanimoto similarity for each test set compound calculated along with the mean of the top k (5) nearest training set calculated using radius=2, matching code for the paper

The `assert np.allclose(...)` check below confirms `splits[...]['y_test']` still matches
Section 4's stored `y_test` for the same (endpoint, featureset) key, i.e. that nothing about the
row alignment changed.

Using the `'hybrid'` featureset as the representative one for the MODEL predictions being
plotted here (change `SIM_FEATURESET` below if you'd rather use `'fcfp4'` or `'rdkit'`) -- this
is separate from the similarity metric itself, which is always FCFP4 regardless.

In [ ]:
TOP_K = 5
SIM_MODELS = ['RF', 'LightGBM', 'MPNN2', 'MPNN3']
PLOT_ENDPOINTS_BAR_MAE_SIMILARITY = ['HLM', 'MDR1', 'RLM', 'SOL']
SIM_FEATURESET = 'hybrid'
SIM_METRICS = ['dice', 'tanimoto']
# MPNN has no 'hybrid' predictions -- it lives under its own graph featureset labels. Every other
# model reads from SIM_FEATURESET. The similarity bins are model-independent (FCFP4 of the test
# compounds), so a model's abs-errors are simply read from wherever that model was evaluated.
SIM_MODEL_FS = {'MPNN1': 'graph', 'MPNN2': 'graph_rdkit', 'MPNN3': 'graph_rdkit2dnorm'}
def sim_fs_for(model):
    return SIM_MODEL_FS.get(model, SIM_FEATURESET)

In [ ]:
def compute_sim_bins(test_fp_indiv, train_fp_full_list, metric='dice', top_k=TOP_K):
        # metric should be dice or tanimoto
        if metric == 'dice':
                sim_fn = DataStructs.BulkDiceSimilarity
        elif metric == 'tanimoto':
                sim_fn = DataStructs.BulkTanimotoSimilarity

        # sim_fn - dependent on dice vs tanimoto, returns a list of similarity scores for the test_fp_indiv against all train_fp_full_list

        sim_all = sim_fn(test_fp_indiv, train_fp_full_list)

        mean_top5_sim = np.mean(sorted(sim_all, reverse=True)[:top_k])

        # sort descending for the sim_scores, takes the top k (5), and computes the mean

        sim_bin = np.clip(np.floor(mean_top5_sim * 10) / 10, 0, 0.9)

        # np.floor(x) rounds down to the nearest integer, so x10 so we dont lose the value, then divide by 10 to rescal back down to the bins lower edge

        return sim_bin


In [ ]:
sim_df_records = []

for ep in MODEL_ENDPOINTS: # Verifying the same SMILES train/test splits produced the corresponding model predictions
    split = splits[(ep, SIM_FEATURESET)]
    smiles_train, smiles_test = split['smiles_train'], split['smiles_test']

    stored_y_test = predictions[(ep, SIM_FEATURESET, 'RF', 'base')]['y_test']

    assert np.allclose(np.asarray(split['y_test']), np.asarray(stored_y_test)), \
    f"y_test misaligned for {ep} -- check splits[(ep, fs)] matches Section 4"
    # checks the two arrays are elementwise equal within floating point tolerance

    train_fps = fcfp4_bit_vectors(smiles_train, radius=2, n_bits=1024, use_features=True)
    test_fps = fcfp4_bit_vectors(smiles_test, radius=2, n_bits=1024, use_features=True)

    for metric in SIM_METRICS:
        sim_bins = np.array([
            compute_sim_bins(fp, train_fps, metric=metric, top_k=TOP_K)
            for fp in test_fps
            ])
        for model in SIM_MODELS:
            pred = predictions[(ep, sim_fs_for(model), model, 'base')]
            abs_errors = np.abs(pred['y_test'] - pred['y_pred_test'])
            for sim_bin, abs_error in zip(sim_bins, abs_errors):
                sim_df_records.append({
                    'endpoint': ep,
                    'metric': metric,
                    'model': model,
                    'sim_bin': sim_bin,
                    'abs_error': abs_error,
                })
                
sim_df = pd.DataFrame(sim_df_records)

In [ ]:
# --- Sanity checks on sim_df ---

# 1. No missing values anywhere (would show up if any dict was missing a key)
print("Nulls per column:")
print(sim_df.isna().sum())

# 2. Exactly the metrics/models/endpoints we expect, nothing stray
print(f"\nUnique metrics: {sorted(sim_df['metric'].unique())}  (expect {SIM_METRICS})")
print(f"Unique models: {sorted(sim_df['model'].unique())}  (expect {sorted(SIM_MODELS)})")
print(f"Unique endpoints: {sorted(sim_df['endpoint'].unique())}  (expect {sorted(MODEL_ENDPOINTS)})")

# 3. sim_bin values should only ever be 0.0, 0.1, ..., 0.9 -- nothing outside that range
print(f"\nsim_bin min/max: {sim_df['sim_bin'].min()}, {sim_df['sim_bin'].max()}  (expect 0.0-0.9)")
print(f"Unique sim_bin values: {sorted(sim_df['sim_bin'].unique())}")

# 4. Row count should be endpoints x metrics x models x (test-set size per endpoint)
n_test_per_ep = {ep: len(splits[(ep, SIM_FEATURESET)]['smiles_test']) for ep in MODEL_ENDPOINTS}
expected_total = sum(n_test_per_ep.values()) * len(SIM_METRICS) * len(SIM_MODELS)
print(f"\nsim_df rows: {len(sim_df)}  (expected: {expected_total})")

# 5. Per (endpoint, metric, model) row counts should each equal that endpoint's test-set size
counts = sim_df.groupby(['endpoint', 'metric', 'model']).size()
print("\nRow counts per (endpoint, metric, model) -- should all match n_test_per_ep for that endpoint:")
display(counts)


In [ ]:
bar_colors = plt.cm.Greys(np.linspace(0.8, 0.3, 10))

In [ ]:
for ep in PLOT_ENDPOINTS_BAR_MAE_SIMILARITY:
    ep_df = sim_df[sim_df['endpoint'] == ep]
    fig, axes = plt.subplots(len(SIM_METRICS), len(SIM_MODELS), figsize=(5 * len(SIM_MODELS), 8), sharex=True, sharey=True, squeeze=False)
    for row, metric in enumerate(SIM_METRICS):
        for col, model_name in enumerate(SIM_MODELS):
            ax = axes[row, col]

            m_df = ep_df[(ep_df['metric'] == metric) & (ep_df['model'] == model_name)]
            stats = m_df.groupby('sim_bin')['abs_error'].agg(['mean', 'std', 'count'])
            stats = stats[stats['count'] > 10]
            stats['sem'] = stats['std'] / np.sqrt(stats['count'])

            for j, (sim_bin, bin_row) in enumerate(stats.iterrows()):
                        ax.bar(j, bin_row['mean'], color=bar_colors[int(round(sim_bin * 10))], width=0.7, alpha=0.95)
                        if bin_row['count'] > 1:
                            _, _, (std_bars,) = ax.errorbar(j, bin_row['mean'], yerr=bin_row['std'], fmt='none', color='black', capsize=3, linewidth=1, label='std')
                            # std answers: how spread out are individual compound errors in this bin
                            std_bars.set_linestyle('dashed')
                            ax.errorbar(j, bin_row['mean'], yerr=bin_row['sem'], fmt='none', color='red', capsize=3, linewidth=1.5, label='SEM')
                            # SEM: standard error of the mean - how confident are we in the bin's mean MAE, SEM shrinks as bin sample size grows
                            ax.text(j, bin_row['mean'], f"n={int(bin_row['count'])}", ha='center', va='bottom', fontsize=7)
            
            ax.set_title(f'{model_name} — {metric}', fontsize=11)

            ax.set_xlabel(f'Mean top-5 {metric} similarity to training set', fontsize=9)
            ax.set_ylabel('Mean absolute error', fontsize=9)

            ax.set_xticks(range(len(stats)))
            ax.set_xticklabels([f'({b:.1f},{b+0.1:.1f}]' for b in stats.index], rotation=45, ha='right', fontsize=8)

            ax.set_ylim(bottom=0)
            ax.grid(True, alpha=0.3, axis='y')

    handles, labels = axes[0, 0].get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    fig.legend(by_label.values(), by_label.keys(), loc='upper right', fontsize=8, ncol=2)

    fig.suptitle(f'Similarity-Binned MAE -- {ep} (mean top-5, featureset={SIM_FEATURESET})', fontsize=11)
    plt.tight_layout()
    plt.savefig(FIGURES / f'section5_sim_binned_mae_{ep}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
for model in SIM_MODELS:
    for metric in SIM_METRICS:
        subset = sim_df[(sim_df['endpoint'] == 'HLM') & (sim_df['model'] == model) & (sim_df['metric'] == metric)]
        total_unfiltered = len(subset)
        stats = subset.groupby('sim_bin')['abs_error'].agg(['count'])
        total_after_filter = stats[stats['count'] > 10]['count'].sum()
        print(f' HLM {model:10s} {metric:10s}  unfiltered N={total_unfiltered}  plotted N (count>10 bins)={total_after_filter}')

for model in SIM_MODELS:
    for metric in SIM_METRICS:
        subset = sim_df[(sim_df['endpoint'] == 'MDR1') & (sim_df['model'] == model) & (sim_df['metric'] == metric)]
        total_unfiltered = len(subset)
        stats = subset.groupby('sim_bin')['abs_error'].agg(['count'])
        total_after_filter = stats[stats['count'] > 10]['count'].sum()
        print(f' RLM {model:10s} {metric:10s}  unfiltered N={total_unfiltered}  plotted N (count>10 bins)={total_after_filter}')

In [ ]:
# Dice and Tanimoto are related by a fixed formula, but it's nonlinear: 
# Dice = 2T / (1 + T), where T is Tanimoto similarity for the same pair of fingerprints. 
# It's monotonic (higher Tanimoto always means higher Dice) and Dice ≥ Tanimoto always 
# — so for any single compound pair, one is just a curved transform of the other.

# Because it's monotonic, the same top-5 nearest neighbours get picked under either metric 
# — ranking a compound's neighbours by Dice vs. by Tanimoto gives the identical top-5 set, 
# just with different similarity values attached.

# --- Prove: same top-5 neighbour SET under Dice vs Tanimoto (rank preservation) ---
ep = 'HLM'
split = splits[(ep, SIM_FEATURESET)]
smiles_train, smiles_test = split['smiles_train'], split['smiles_test']

train_fps = fcfp4_bit_vectors(smiles_train, radius=2, n_bits=1024, use_features=True)
test_fps = fcfp4_bit_vectors(smiles_test, radius=2, n_bits=1024, use_features=True)

matches = []
mean_top5_tanimoto = []
std_top5_tanimoto = []
for fp in test_fps:
    dice_sims = np.array(DataStructs.BulkDiceSimilarity(fp, train_fps))
    tanimoto_sims = np.array(DataStructs.BulkTanimotoSimilarity(fp, train_fps))

    top5_dice_idx = np.argsort(dice_sims)[::-1][:5]
    top5_tanimoto_idx = np.argsort(tanimoto_sims)[::-1][:5]

    matches.append(set(top5_dice_idx) == set(top5_tanimoto_idx))
    top5_tanimoto_vals = tanimoto_sims[top5_tanimoto_idx]
    mean_top5_tanimoto.append(top5_tanimoto_vals.mean())
    std_top5_tanimoto.append(top5_tanimoto_vals.std())

mean_top5_tanimoto = np.array(mean_top5_tanimoto)
std_top5_tanimoto = np.array(std_top5_tanimoto)

n_match = sum(matches)
print(f'{ep}: top-5 neighbour SET identical under Dice vs Tanimoto for '
      f'{n_match}/{len(matches)} test compounds ({100 * n_match / len(matches):.1f}%)')

# But mean() doesn't commute with a nonlinear function. 
# mean(top-5 Dice) ≠ 2 × mean(top-5 Tanimoto) / (1 + mean(top-5 Tanimoto)). 
# Since Dice is concave in Tanimoto (its slope flattens as T grows), 
# a compound whose 5 nearest-neighbour similarities are spread out gets pulled down more by this transform 
# than a compound whose 5 are tightly clustered — even if both have the same mean Tanimoto. 
# That can flip which side of a 0.1 bin-boundary a compound lands on, purely due to this aggregation effect, 
# not because it moved rank relative to other compounds.

# Net effect: individual compounds can cross a fixed-width bin boundary 
# under one metric but not the other, even though the underlying similarity structure (who's near whom) hasn't changed. 
# That's exactly why your Dice row and Tanimoto row don't produce identical-looking bars 
# — it's a real property of the metrics, not an error in compute_sim_bins().

# --- Concrete example: two compounds with similar mean Tanimoto, different spread ---

# Find one "tight cluster" example and one "spread out" example, matched on similar mean_top5_tanimoto
# so it's a fair, apples-to-apples comparison

target_mean = 0.4
close_to_target = np.abs(mean_top5_tanimoto - target_mean) < 0.03

tight_idx = np.where(close_to_target)[0][np.argmin(std_top5_tanimoto[close_to_target])]
spread_idx = np.where(close_to_target)[0][np.argmax(std_top5_tanimoto[close_to_target])]

for label, i in [('TIGHT cluster', tight_idx), ('SPREAD OUT cluster', spread_idx)]:
    fp = test_fps[i]
    tanimoto_sims = np.array(DataStructs.BulkTanimotoSimilarity(fp, train_fps))
    dice_sims = np.array(DataStructs.BulkDiceSimilarity(fp, train_fps))
    top5_idx = np.argsort(tanimoto_sims)[::-1][:5]

    top5_tanimoto = tanimoto_sims[top5_idx]
    top5_dice = dice_sims[top5_idx]

    mean_t = top5_tanimoto.mean()
    mean_d = top5_dice.mean()
    predicted_d = 2 * mean_t / (1 + mean_t)

    print(f'--- {label} (compound {i}) ---')
    print(f'  top-5 Tanimoto values: {np.round(top5_tanimoto, 3)}  (std={top5_tanimoto.std():.3f})')
    print(f'  top-5 Dice values:     {np.round(top5_dice, 3)}')
    print(f'  mean(top-5 Tanimoto)          = {mean_t:.4f}  -> bin {np.floor(mean_t*10)/10:.1f}')
    print(f'  actual mean(top-5 Dice)       = {mean_d:.4f}  -> bin {np.floor(mean_d*10)/10:.1f}')
    print(f'  predicted Dice (2T/(1+T) on mean_t) = {predicted_d:.4f}')
    print(f'  gap (actual - predicted) = {mean_d - predicted_d:.4f}\n')

> 📝 **REVIEW (NOTEBOOK_REVIEW.md #21):** code-quality pass wanted — can this be simplified / vars reused? (Logic is fine.)

### 5.5 -- ANOVA + Tukey HSD post-hoc, per endpoint (items 2 & 3)

Analysis done for the `hybrid` featureset (fcfp4 + RDKit descriptors combined) -- it dominates or ties
fcfp4/rdkit alone for every model (Section 5.2), so it's the fair single representation to compare models on and likely what they did in the paper (although couldnt find it directly stated).

One ANOVA + Tukey HSD per endpoint, using the 15 `RepeatedKFold` CV-fold Pearson r scores
(persisted per model in Section 4.2's `predictions[...]['cv_scores']`) as the paired "subjects".

For a given endpoint, say HLM, a single ANOVA call answers one question, across RF, SVM, XGBoost and LightGBM,
is there a statistically significant difference in mean Pearson r? Will call 4x (per endpoint) for all cv_pearson r, where each model was used to determine a cv pearson r for each fold (15*4). It's comparing RF-vs-SVM-vs-XGBoost-vs-LightGBM within each fold's own frame of reference, then checking whether that within-fold pattern holds up consistently across all 15 folds.

Works because `model_validation()` reuses the same `RepeatedKFold(random_state=128)` split
for every model on a given (endpoint, featureset) -- so fold 0 is the same train/val partition
for RF, SVM, XGBoost, and LightGBM alike, making a fold a repeated-measurement
(`AnovaRM`, subject=fold, within=model). 

Heatmap colour is binned by significance level (NS / p<0.05 / p<0.01 / p<0.001).

`pairwise_tukeyhsd`'s summary table is each
heatmap's data source and the "differences in Pearson r values" table (item 3).

In [ ]:
HYBRID_FEATURESET = 'hybrid'
HEATMAP_MODEL_LIST = ['RF', 'SVM', 'XGBoost', 'LightGBM','FCNN','MPNN1', 'MPNN2', 'MPNN3']  # aspirational full set
HEATMAP_EP_LIST = ['HLM', 'MDR1', 'SOL', 'RLM']
arm = 'base'

# MPNN lives under its own featureset labels; every other model uses the hybrid representation.
HEATMAP_MODEL_FS = {'MPNN1': 'graph', 'MPNN2': 'graph_rdkit', 'MPNN3': 'graph_rdkit2dnorm'}


def build_cv_long_df(HYBRID_FEATURESET, HEATMAP_MODEL_LIST, HEATMAP_EP_LIST, predictions,
                     arm='base', model_fs=None, expected_folds=None):
    """Long-format CV Pearson r per (model, endpoint, fold), keeping ONLY models present on the
    shared fold grid across every endpoint. AnovaRM (subject=fold, within=model) needs a balanced,
    *paired* design -- fold i must be the same RepeatedKFold partition for every model. That holds
    for models run through model_validation() with the same RepeatedKFold(n_splits=5, n_repeats=3,
    random_state=128); a model with a different fold count (e.g. MPNN in 'sample' mode) is unpaired
    and is dropped here so the ANOVA stays valid. Returns (cv_long_df, kept_models). `model_fs`
    maps a model -> its featureset label (default HYBRID_FEATURESET)."""
    model_fs = model_fs or {}

    def scores(ep, m):
        key = (ep, model_fs.get(m, HYBRID_FEATURESET), m, arm)
        return np.asarray(predictions[key]['cv_scores']) if key in predictions else None

    if expected_folds is None:  # reference grid = the most common fold count actually present
        lengths = [len(s) for ep in HEATMAP_EP_LIST for m in HEATMAP_MODEL_LIST
                   if (s := scores(ep, m)) is not None]
        expected_folds = max(set(lengths), key=lengths.count) if lengths else 0

    kept, dropped, records = [], [], []
    for m in HEATMAP_MODEL_LIST:
        reason = None
        for ep in HEATMAP_EP_LIST:
            s = scores(ep, m)
            if s is None:
                reason = f'no prediction at {ep}'; break
            if len(s) != expected_folds:
                reason = f'{len(s)} folds != {expected_folds} at {ep} (unpaired; needs same RepeatedKFold)'; break
        if reason:
            dropped.append((m, reason)); continue
        kept.append(m)
        for ep in HEATMAP_EP_LIST:
            for fold_id, r in enumerate(scores(ep, m)):
                records.append({'endpoint': ep, 'model': m, 'cv_pearson_r': r, 'fold_id': fold_id})

    for m, reason in dropped:
        print(f'build_cv_long_df: EXCLUDED {m} -- {reason}')
    print(f'build_cv_long_df: {len(kept)} models on the shared {expected_folds}-fold grid: {kept}')
    return pd.DataFrame(records), kept

In [ ]:
cv_long_df, ANOVA_MODELS = build_cv_long_df(
    HYBRID_FEATURESET, HEATMAP_MODEL_LIST, HEATMAP_EP_LIST, predictions, model_fs=HEATMAP_MODEL_FS)
cv_long_df.head(16)

In [ ]:
# Check normality of cv_pearson_r for one endpoint (assumption behind AnovaRM).
# Change NORM_CHECK_EP to inspect SOL / RLM / MDR1 -- values are deterministic (fixed RepeatedKFold
# seed, loaded from the section-4 checkpoint), so they don't change across kernel restarts.
NORM_CHECK_EP = 'MDR1'
norm_check_df = cv_long_df[cv_long_df['endpoint'] == NORM_CHECK_EP]

for model in ANOVA_MODELS:
    r_vals = norm_check_df[norm_check_df['model'] == model]['cv_pearson_r']
    shapiro_stat, shapiro_p = stats.shapiro(r_vals)
    print(f"{model:10s} Shapiro-Wilk: W={shapiro_stat:.3f}, p={shapiro_p:.3f} "
          f"({'normal' if shapiro_p > 0.05 else 'NOT normal'} at alpha=0.05)")

fig, axes = plt.subplots(1, len(ANOVA_MODELS), figsize=(4 * len(ANOVA_MODELS), 4))
for ax, model in zip(axes, ANOVA_MODELS):
    r_vals = norm_check_df[norm_check_df['model'] == model]['cv_pearson_r']
    stats.probplot(r_vals, dist='norm', plot=ax)
    ax.set_title(f'{NORM_CHECK_EP} - {model}')

plt.tight_layout()
plt.show()

# Fail to reject null hypothesis (sample drawn from a normal distribution)

**Quantifying *how far* from normal.** Shapiro-Wilk only gives a reject/keep verdict on H0 = "normal" and has low power at n=15, so it doesn't say *how far* off a distribution is. Below we take the one group we flagged — **RF at MDR1** — and describe its shape with **skew** and **excess kurtosis** (both 0 for a true Gaussian), plus a histogram against the fitted normal PDF and a Q-Q plot. (HLM/MPNN2 was the other flagged case but is not pursued — that data is being migrated.)

In [ ]:
# Is RF's fold-to-fold performance on MDR1 normally distributed?
# Show the 15 CV Pearson r values as a histogram, with a normal curve of the same mean/spread.
from scipy import stats  # re-import guards against `stats` being shadowed by an earlier cell

r = cv_long_df.query("endpoint == 'MDR1' and model == 'RF'")['cv_pearson_r']

p    = stats.shapiro(r).pvalue
skew = stats.skew(r)
kurt = stats.kurtosis(r)          # Fisher: 0 for a true Gaussian

# normal curve with the same mean and spread as the data
x = np.linspace(r.min() - 0.05, r.max() + 0.05, 200)
normal_curve = stats.norm.pdf(x, r.mean(), r.std(ddof=1))

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(r, bins=8, density=True, color='steelblue', edgecolor='white', label='CV Pearson r (15 folds)')
ax.plot(x, normal_curve, 'r-', lw=2, label='normal curve (same mean & spread)')
ax.set_xlabel('CV Pearson r')
ax.set_ylabel('density')
ax.set_title(f'MDR1 - RF   |   Shapiro p={p:.3f}   skew={skew:+.2f}   excess kurt={kurt:+.2f}')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
anova_results = {}  # anova_results[ep] = (F value, p value) -- created once, before the loop
tukey_tables = {}

for ep in HEATMAP_EP_LIST:
    ep_df = cv_long_df[cv_long_df['endpoint'] == ep ]

    # call ANOVA on each endpoint, HLM, RLM etc
    anova_result = AnovaRM(ep_df, depvar='cv_pearson_r', subject='fold_id', within=['model']).fit() # One factor = one-way the single factor being within=['model']

    f_val = anova_result.anova_table['F Value'].iloc[0]
    p_val = anova_result.anova_table['Pr > F'].iloc[0]
    anova_results[ep] = (f_val, p_val)

    # For a given endpoint, say HLM, a single ANOVA call answers one question, across RF, SVM, XGBoost and LightGBM,
    # is there a statistically significant difference in mean Pearson r?
    # Will call 4x (per endpoint) for all cv_pearson r, where each model was used to determine a cv pearson r for each fold (15*4)

    tukey = pairwise_tukeyhsd(endog=ep_df['cv_pearson_r'], groups=ep_df['model'], alpha=0.05)
    # endog = dependent var, array, every individual scross across all groups, not aggregated
    # groups = array like of the same length as endog, giving each obsv group label (model col), pairs with endog by positions
    tukey_tables[ep] = pd.DataFrame(tukey.summary().data[1:], columns=tukey.summary().data[0])

display(ep_df.head())
display(ep, anova_result.anova_table)

# Num DF (Numerator degrees of Freedom being tested, how many independent ways can the 4 model means differ from each other), Formula is (number of models) - 1
# Den DF (Denominator degrees of freedom) - the degrees of fredom for the residual/error term i.e the leftover variance not explained by model choice, for a one-way repeated measures design,
# Formula is (number of subjects - 1) x (number of levels -1) = (15 - 1) x (4 - 1) = 14 x 3 = 42
# Reflect how much independent information is left over to estimate "noise" after accounting for both fold-to-fold variation
# and model-to-model variation

# F values is a ratio: var explained by model choice / leftover residual variance, if model choice made no real diff,
# would expect this to hover around 1 (explained var aprox noise var).
# Getting 69.7 means var between models is 70x larger than what id expect from noise along, large not subtle signal

# Pr > F is the p-value, given Num DF=3, and Den DF=42, 

# H0 (null hypoth): All 4 models have the same true mean person r for this endpoint, μ_RF = μ_SVM etc
# H1 (Alt hypoth): Reject null hypothesis, at least one models true mean Pearson r differs from at least one others
# Anova is the omnibus, due to rejecting the null, we can apply the Tukey

# Separate loop, after all endpoints are processed and stored
for ep, (f_val, p_val) in anova_results.items():
    verdict = "significant (reject H0)" if p_val < 0.05 else "not significant (fail to reject H0)"
    print(f"{ep:5s}  F={f_val:.2f}  p={p_val:.3g}  -->  {verdict}")

for ep in HEATMAP_EP_LIST:
    print(f"\n--- {ep} ---")
    print(tukey_tables[ep].to_string(index=False))

# Example H0/H1 for one specific Tukey row — using my actual HLM output, 
# the LightGBM vs RF row (meandiff = -0.039, p-adj = 0.0, reject = True):

# H0: μ_LightGBM = μ_RF — LightGBM and RF have the same true mean Pearson r for HLM. 
# H1: μ_LightGBM ≠ μ_RF — they differ (this is a two-sided test; 
# Tukey doesn't ask "which one is better," just "are they different").

**Caution — Tukey HSD assumptions at MDR1/RF.** Tukey HSD assumes normal, equal-variance groups. The Section 5.5 normality check flagged **RF at MDR1** as non-normal: right-skewed (skew +1.56) with a heavy tail (excess kurtosis +2.25) driven by a single high outlier fold (≈0.76), which inflates that group's variance. So the pairwise **p-adj values for rows involving RF at MDR1** are the least reliable in the heatmap and should be read with caution. This does **not** invalidate the MDR1 omnibus ANOVA (balanced design + large F make RM-ANOVA robust to one mildly non-normal group) — it is a localised note on specific Tukey comparisons only. (The other flagged group, HLM/MPNN2, is not pursued — that data is being migrated.) See ADR-010 in DECISIONS.md.

In [ ]:
pd.cut(tukey_tables[ep]['p-adj'], [0, 0.05, 1], right=False, labels= [True, False]).value_counts()

In [ ]:
# empty cell

sig_counts = {}

for ep in HEATMAP_EP_LIST:
    value = pd.cut(tukey_tables[ep]['p-adj'], [0, 0.05, np.inf], right=False, labels= ['Signif', 'NS']).value_counts()
    sig_counts.update({ep: value})

pd.DataFrame(sig_counts)

HLM: Their heatmap vs mine matches

MDR1 : As above

SOL: Doesnt match, I see 3 sig 3 NS they have : 1 SF, 5 NS

RLM: No match, I see 4 SF and 2 NS, they have 5 SF, 1 NS

In [ ]:
tukey_tables['SOL']

In [ ]:
diff_matrices = {}  # diff_matrices[ep] = 4x4 DataFrame of meandiff
padj_matrices = {}  # padj_matrices[ep] = 4x4 DataFrame of p-adj

for ep in HEATMAP_EP_LIST:
    diff_matrix = pd.DataFrame(np.nan, index=ANOVA_MODELS, columns=ANOVA_MODELS, dtype=float)
    padj_matrix = pd.DataFrame(np.nan, index=ANOVA_MODELS, columns=ANOVA_MODELS, dtype=float)

    for _, row in tukey_tables[ep].iterrows():
        g1, g2 = row['group1'], row['group2']
        diff_matrix.loc[g1, g2] = row['meandiff']
        diff_matrix.loc[g2, g1] = -row['meandiff']
        padj_matrix.loc[g1, g2] = row['p-adj']
        padj_matrix.loc[g2, g1] = row['p-adj']

    diff_matrices[ep] = diff_matrix
    padj_matrices[ep] = padj_matrix

display(diff_matrices)
display(padj_matrices)

In [ ]:
SIG_COLORS = ['#f7c6d0', '#c8e6c9', '#66bb6a', '#1b5e20']  # NS, p<0.05, p<0.01, p<0.001
SIG_LABELS = ['NS', 'p<0.05', 'p<0.01', 'p<0.001']
sig_cmap = ListedColormap(SIG_COLORS)
sig_norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], sig_cmap.N)
legend_handles = [Patch(facecolor=c, edgecolor='black', label=l) for c, l in zip(SIG_COLORS, SIG_LABELS)]

def _sig_code(p):
    if pd.isna(p):
        return np.nan
    if p < 0.001:
        return 3
    elif p < 0.01:
        return 2
    elif p < 0.05:
        return 1
    return 0

def _sig_label(p):
    return '' if pd.isna(p) else SIG_LABELS[int(_sig_code(p))]

fig, axes = plt.subplots(2, 2, figsize=(13, 11))
axes = axes.flatten()

for ax, ep in zip(axes, HEATMAP_EP_LIST):
    diff_matrix = diff_matrices[ep]
    padj_matrix = padj_matrices[ep]
    f_val, p_val = anova_results[ep]
    anova_str = f"ANOVA p={p_val:.3f}" if p_val >= 0.001 else "ANOVA p<0.001"

    sig_code_matrix = padj_matrix.map(_sig_code)  # continuous p-adj -> 0/1/2/3 significance tier

    annot_matrix = pd.DataFrame('', index=ANOVA_MODELS, columns=ANOVA_MODELS)
    for m1 in ANOVA_MODELS:
        for m2 in ANOVA_MODELS:
            if m1 != m2:
                annot_matrix.loc[m1, m2] = f"{diff_matrix.loc[m1, m2]:.3f}\n{_sig_label(padj_matrix.loc[m1, m2])}"

    sns.heatmap(sig_code_matrix, cmap=sig_cmap, norm=sig_norm, annot=annot_matrix, fmt='',
                mask=diff_matrix.isna(), ax=ax, cbar=False, annot_kws={'fontsize': 8})
    
    n_folds = cv_long_df["fold_id"].nunique()
    ax.set_title(f"{ep} -- {anova_str}\n(hybrid featureset, {n_folds} paired CV folds)", fontsize=10)
    ax.legend(handles=legend_handles, loc='center left', bbox_to_anchor=(1.02, 0.5),
          fontsize=7, borderaxespad=0)


fig.suptitle('Pairwise CV-fold Pearson r comparisons (Tukey HSD), per endpoint -- hybrid featureset\n'
             'cell = meandiff (row - col), coloured by significance', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES / 'section5_tukey_heatmap_per_endpoint.png', dpi=150, bbox_inches='tight')
plt.show()


> 📝 **REVIEW (NOTEBOOK_REVIEW.md #21):** code-quality pass wanted — can this be simplified / vars reused? (Logic is fine.)

### 5.6 — Figure 7: effect of hyperparameter tuning (default vs optimized)

Recreates the paper's Figure 7 — a 1×3 grid of grouped bar charts for the representative models **RF, LightGBM, MPNN2** (the trio that fuse fingerprints + descriptors). Per subplot: x-axis = endpoints, two bars each — **default** (`base` arm) vs **optimized** (`tuned` arm) — y-axis = test-set Pearson r.

- **Needs the `tuned` arm** (Section 4.4). Structure is built now; the *optimized* bars stay empty until the tuning run populates the `tuned` keys, then this cell fills automatically.
- RF/LightGBM use `FIG7_CLASSICAL_FS` (default `hybrid`); MPNN2 uses `graph_rdkit`. hPPB/rPPB omitted (unmodelled).
- **Error bars**: the paper puts error bars on a test-set r but doesn't state the source (a single test set yields one r). We use a **bootstrap** over the test compounds (resample with replacement, `N_BOOT` times → SD of the test-set Pearson r) — a genuine uncertainty for the single test metric. Set `N_BOOT=0` to drop them.

In [ ]:
# --- Figure 7: default (base) vs optimized (tuned) test-set Pearson r, per representative model ---
FIG7_CLASSICAL_FS = 'hybrid'          # RF/LightGBM featureset (MPNN2 uses its own graph_rdkit)
FIG7_MODELS = ['RF', 'LightGBM', 'MPNN2', 'MPNN3']
FIG7_MODEL_FS = {'RF': FIG7_CLASSICAL_FS, 'LightGBM': FIG7_CLASSICAL_FS, 'MPNN2': 'graph_rdkit', 'MPNN3': 'graph_rdkit2dnorm'}
FIG7_ENDPOINTS = ['HLM', 'MDR1', 'SOL', 'RLM']    # hPPB/rPPB omitted (not modelled)
FIG7_ARMS = [('base', 'default', 'black'), ('tuned', 'optimized', 'darkgrey')]
N_BOOT = 1000                          # bootstrap resamples for the test-r error bar (0 = no error bars)

def _fig7_test_r(ep, model, arm):
    """(test-set Pearson r, bootstrap SD) from stored test predictions; (nan, nan) if key absent."""
    pred = predictions.get((ep, FIG7_MODEL_FS[model], model, arm))
    if pred is None:
        return np.nan, np.nan
    yt = np.asarray(pred['y_test'], float)
    yp = np.asarray(pred['y_pred_test'], float)
    r = np.corrcoef(yt, yp)[0, 1]
    if not N_BOOT:
        return r, np.nan
    rng = np.random.default_rng(0)
    n = len(yt)
    boots = [np.corrcoef(yt[idx], yp[idx])[0, 1] for idx in (rng.integers(0, n, n) for _ in range(N_BOOT))]
    return r, float(np.nanstd(boots))

fig, axes = plt.subplots(1, len(FIG7_MODELS), figsize=(5 * len(FIG7_MODELS), 4), sharey=True)
x = np.arange(len(FIG7_ENDPOINTS))
width = 0.38
for ax, model in zip(axes, FIG7_MODELS):
    for k, (arm, label, color) in enumerate(FIG7_ARMS):
        stats = [_fig7_test_r(ep, model, arm) for ep in FIG7_ENDPOINTS]
        heights = [s[0] for s in stats]
        errs = [s[1] for s in stats]
        ax.bar(x + (k - 0.5) * width, heights, width, yerr=errs, capsize=3,
               color=color, edgecolor='black', label=label, error_kw={'elinewidth': 1})
    ax.set_title(model, fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(FIG7_ENDPOINTS, rotation=30, ha='right')
    ax.set_ylim(0, 1.0)
    ax.set_ylabel("Pearson's r for test set")
    ax.grid(alpha=0.3, axis='y')
    ax.legend(fontsize=9)

fig.suptitle('Fig 7 -- Effect of hyperparameter tuning on representative ML models (test-set Pearson r)', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES / 'section5_fig7_tuning_effect.png', dpi=150, bbox_inches='tight')
plt.show()

> 📝 **REVIEW (NOTEBOOK_REVIEW.md #21):** code-quality pass wanted — can this be simplified / vars reused? (Logic is fine.)

### 5.7 — Table 2: Summary statistics + model Pearson r (paper's Table 2)

Recreates the paper's Table 2. The **top block** is the Section 2.1 summary statistics; the **bottom block** is per-model Pearson r as `CV_r (test_r)` — the value without parentheses is cross-validation on the 80% training set, the value in parentheses is the held-out 20% test set.

- Row suffix **ᵇ = before tuning** (`base` arm) · **ᶜ = after tuning** (`tuned` arm — blank until Section 4.4 runs).
- Models: RF, LightGBM, MPNN1, MPNN2. RF/LightGBM read `TABLE2_CLASSICAL_FS` (default `hybrid`); MPNN1/MPNN2 read their own `graph`/`graph_rdkit` featuresets.
- **Deviations from the paper**: (1) our CV_r is the mean of `RepeatedKFold(5×3)` = 15 folds, a steadier estimate than the paper's single 5-fold; (2) **hPPB/rPPB show stats only** — those endpoints aren't modelled here, so their model-r cells stay blank; (3) MPNN ᵇ values reflect whatever `MPNN_MODE` produced them (sample until the full run).

In [ ]:
# --- Table 2: §2.1 summary stats (top) + model Pearson r before/after tuning (bottom) ---
# Cell 'CV_r (test_r)': CV_r = mean RepeatedKFold(5x3) Pearson r on the 80% training set (we report
# the mean of 15 folds -- steadier than the paper's single 5-fold); test_r = Pearson r on the 20%
# test set. Row suffix b = before tuning ('base' arm), c = after tuning ('tuned' arm, blank until 4.4).
TABLE2_CLASSICAL_FS = 'hybrid'   # featureset for the RF/LightGBM rows (MPNN uses its own graph fs)
TABLE2_MODELS = ['RF', 'LightGBM', 'MPNN1', 'MPNN2', 'MPNN3']
TABLE2_MODEL_FS = {'RF': TABLE2_CLASSICAL_FS, 'LightGBM': TABLE2_CLASSICAL_FS,
                   'MPNN1': 'graph', 'MPNN2': 'graph_rdkit', 'MPNN3': 'graph_rdkit2dnorm'}
ARM_SUFFIX = {'base': 'ᵇ', 'tuned': 'ᶜ'}

# top half -- identical to Section 2.1's summary statistics (all 6 endpoints)
table2_top = df[ENDPOINT_COLS].agg(['count', 'min', 'max', 'mean', 'median', 'std'])
table2_top.columns = list(ENDPOINTS.keys())
table2_top.index = ['Compounds', 'min', 'max', 'mean', 'median', 'SD']
table2_top = table2_top.round(2).astype(object)

# bottom half -- 'CV_r (test_r)' per model x arm; blank where no data (PPB endpoints, or untuned 'c')
def _table2_cell(ep, model, arm):
    fs = TABLE2_MODEL_FS[model]
    r = results_df[(results_df['endpoint'] == ep) & (results_df['featureset'] == fs)
                   & (results_df['model'] == model) & (results_df['arm'] == arm)]
    if r.empty or pd.isna(r['Pearson_r_CV'].iloc[0]):
        return ''
    return f"{r['Pearson_r_CV'].iloc[0]:.2f} ({r['Pearson_r_test'].iloc[0]:.2f})"

table2_bottom = pd.DataFrame(
    {ep: {f'R ({m}){ARM_SUFFIX[arm]}': _table2_cell(ep, m, arm)
          for m in TABLE2_MODELS for arm in ('base', 'tuned')}
     for ep in ENDPOINTS.keys()}
)

table2 = pd.concat([table2_top, table2_bottom])
table2